# Private Pre-emption: numerical workbook

This workbook accompanies **Pre-empting Competitive Sales: Private Offers, Early Resolution, and Seller Information**. It reproduces the numerical examples in the main paper and online appendix, then separates out additional research diagnostics. Start with the participation example and the welfare comparison below; expand the calculation records to inspect every input script and its full output.

[Main paper](Manuscript-short.pdf) · [Online appendix](Manuscript-online-appendix.pdf) · [Download the minimal reproduction package](pre-emption-numerics.zip) · [Download the executed notebook](pre-emption-numerics.ipynb)

The ZIP contains only the runnable code, configuration, pinned dependencies, expected figure sources, and a readable notebook. It deliberately excludes the PDFs, recorded logs, and supporting notes. **Download and extract the ZIP to rerun the calculations**; the notebook alone does not contain its supporting scripts. Open `pre-emption-numerics.ipynb` in Jupyter or VS Code and run all cells from the top. The web version is a saved execution, not an interactive server.

## 1. Reproduce the calculations

Use Python 3.12. From the extracted package directory, install `code/requirements-workbook.txt` with pip. A fresh run starts in the next cell, writes a new timestamped result directory, and stops on failure. No manuscript or figure source is overwritten. The command-line alternative, requiring only the two numerical dependencies in `code/requirements-publication.txt`, is `python code/run_reproduction.py --mode publication`.

Exact arithmetic checks identities and numerical premises. Floating-point audits check calibrations and convergence. Interval certificates enclose rounding and integration error and establish the stated numerical inequalities. These checks accompany the analytical proofs; they do not establish global equilibrium uniqueness. The research diagnostics at the end are explicitly outside that certification claim.

## Explore the core trade-off

This is an illustrative calculator, not an equilibrium solver. It lets you vary the contracting parties' resolution benefits and the competition wedge. The displayed net joint gain is \(d_B+d_S-\Gamma-\kappa\): early agreement is privately feasible when it is positive. It says nothing by itself about the equilibrium offer price, acceptance probability, or social desirability.

<section class="explorer" aria-label="Resolution and competition calculator">
  <div class="explorer-controls">
    <label>Buyer resolution benefit <output id="buyer-benefit-value">0.08</output><input id="buyer-benefit" type="range" min="0" max="0.20" step="0.01" value="0.08"></label>
    <label>Seller resolution benefit <output id="seller-benefit-value">0.03</output><input id="seller-benefit" type="range" min="0" max="0.20" step="0.01" value="0.03"></label>
    <label>Competition wedge <span class="formula">Γ</span> <output id="wedge-value">0.07</output><input id="wedge" type="range" min="0" max="0.20" step="0.01" value="0.07"></label>
    <label>Offer cost <span class="formula">κ</span> <output id="cost-value">0.02</output><input id="cost" type="range" min="0" max="0.10" step="0.01" value="0.02"></label>
  </div>
  <div class="explorer-result" aria-live="polite"><strong id="private-result">Private joint gain: 0.02</strong><span id="private-explanation">A mutually beneficial early agreement can be feasible.</span></div>
  <p class="explorer-note">The wedge is the value of the later competitive process foregone by the buyer and seller together. A higher late buyer can raise allocative value even where the contracting parties still prefer early agreement.</p>
</section>

In [1]:
from pathlib import Path
import ast, hashlib, html, importlib.metadata, json, os, re, subprocess, sys
from datetime import datetime, timezone
from IPython.display import HTML, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'code/claims.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Extract the complete ZIP and open the notebook there.')
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Use Python 3.12 for the recorded reproduction environment.')
for package, expected in [('numpy', '2.3.5'), ('python-flint', '0.9.0')]:
    if importlib.metadata.version(package) != expected:
        raise RuntimeError(f'Install {package}=={expected} before continuing.')

def table(headers, rows):
    esc = lambda x: html.escape(str(x))
    head = ''.join('<th>' + esc(x) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + esc(x) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<div class="table-wrap"><table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table></div>'))

# Remove optimization settings so the audits' assertions remain active.
env = os.environ.copy()
env.pop('PYTHONOPTIMIZE', None)
env['PYTHONUTF8'] = '1'
name = datetime.now(timezone.utc).strftime('workbook-%Y%m%dT%H%M%S.%fZ')
RUN = ROOT / 'code/reproduction_outputs' / name
command = [sys.executable, '-X', 'utf8', str(ROOT / 'code/run_reproduction.py'),
           '--mode', 'publication', '--run-name', name]
completed = subprocess.run(command, cwd=ROOT, env=env, capture_output=True,
                           text=True, encoding='utf-8', timeout=3600)
if completed.returncode:
    raise RuntimeError(completed.stdout + '\n' + completed.stderr)
summary = json.loads((RUN / 'summary.json').read_text(encoding='utf-8'))
manifest = json.loads((RUN / 'claims.json').read_text(encoding='utf-8'))
tasks = {t['id']: t for t in summary['tasks']}
specs = {t['id']: t for t in manifest['tasks']}
if summary['status'] != 'pass' or set(tasks) != set(specs):
    raise RuntimeError('The complete suite did not finish successfully.')
for record in json.loads((RUN / 'source_hashes.json').read_text(encoding='utf-8')):
    if hashlib.sha256((ROOT / record['path']).read_bytes()).hexdigest() != record['sha256']:
        raise RuntimeError('A source changed during execution: ' + record['path'])

shown = set()
def records(ids):
    for task_id in ids:
        shown.add(task_id)
        task, spec = tasks[task_id], specs[task_id]
        scripts = [spec['script']] if spec['kind'] == 'python' else spec['scripts']
        output = (RUN / (task_id + '.stdout.txt')).read_text(encoding='utf-8')
        stderr = (RUN / (task_id + '.stderr.txt')).read_text(encoding='utf-8')
        if stderr.strip():
            output += '\nSTDERR\n' + stderr
        title = f"{task['title']} | {task['classification']} | {task['status']}"
        display(HTML('<details class="record"><summary>' + html.escape(title) + '</summary><p><code>'
                     + html.escape(', '.join(scripts)) + '</code></p><pre>'
                     + html.escape(output) + '</pre></details>'))

table(['Execution', 'Recorded value'], [
    ['Completed (UTC)', summary['finished_at_utc']],
    ['Checks passed', f"{len(tasks)} / {len(specs)}"],
    ['Python', sys.version.split()[0]],
    ['Numerical packages', 'numpy 2.3.5; python-flint 0.9.0'],
    ['Compute time', f"{sum(t['duration_seconds'] for t in tasks.values()):.1f} seconds"],
])

Execution,Recorded value
Completed (UTC),2026-09-20T09:41:44.968710+00:00
Checks passed,22 / 22
Python,3.12.14
Numerical packages,numpy 2.3.5; python-flint 0.9.0
Compute time,200.6 seconds


## 2. Participation information is enough for probing

**Main paper, Section 3.2 and Proposition 3.1.** Values are uniform on [0,1]. Only the number of later buyers changes with the seller's state: 2 or 6. The seller's value is 0.4 in both states, the high-state probability is 0.5, the offer cost is 0.02, and the buyer's resolution benefit has support [0,0.1]. Seller resolution benefits are zero.

The sufficient probing window contains 0.1. Thus differences in expected competition alone can support both accepted and rejected offers. The distribution of the buyer's resolution benefit is otherwise unspecified within the paper's maintained class. Consequently this example supplies sufficient conditions, not a numerical offer price or attempt rate.

In [2]:
participation = json.loads((RUN / 'participation_only_exact.json').read_text())['participation']
table(['Quantity', 'Exact fraction', 'Decimal'], [
    [name, value['exact'], f"{value['decimal']:.12f}"]
    for name, value in participation['quantities'].items()
])
records(['participation_only_exact'])

Quantity,Exact fraction,Decimal
C_L(0),59/125,0.472000000000
Delta_D,70753/1093750,0.064688457143
bar_D_H,114503/1093750,0.104688457143
bar_D_L,1/25,0.040000000000
lower_type_condition,64/125,0.512000000000
probing_window_upper,92628/546875,0.169376914286
w_H(1)=C_H(1),468878/546875,0.857376914286
w_L(1)=C_L(1),86/125,0.688000000000


## 3. A ban can raise or lower welfare

**Main paper, Table 4.1 and Appendix D.** The two calibrations differ only in the common distribution of early and late buyer values. Both have mean 0.5. The first concentrates values near the mean; the second gives more weight to the upper tail. The equilibrium prices and the types making early offers change with the distribution.

The welfare effect is **avoided offer costs + improved allocation − forgone resolution gains**. Negative values mean that banning pre-emption lowers welfare. These examples establish opposite rankings, not a general comparative-statics result. Their seller values differ across states, unlike the participation-only example above. Full primitives and convergence checks are in the expanded record.

In [3]:
raw = (RUN / 'ban_welfare_examples_floating.stdout.txt').read_text()
cases = [raw.split('SYMMETRIC\n', 1)[1].split('UPPER_TAIL\n', 1)[0],
         raw.split('UPPER_TAIL\n', 1)[1]]
def field(text, name):
    line = next(line for line in text.splitlines() if line.startswith(name + ' '))
    return ast.literal_eval(line[len(name) + 1:])
prices = [field(text, 'q') for text in cases]
outcomes = [field(text, 'outcomes') for text in cases]
welfare = [field(text, 'welfare') for text in cases]
rows = [
    ['Probing price', prices[0][0], prices[1][0]],
    ['Knockout price', prices[0][1], prices[1][1]],
    ['Rejected attempt', outcomes[0][1], outcomes[1][1]],
    ['Early agreement', outcomes[0][2], outcomes[1][2]],
]
for label, key in [('Avoided offer costs', 'attempt_resources'),
                   ('Allocative gain', 'allocation_gain'),
                   ('Forgone resolution gains', 'timing_costs'),
                   ('Welfare effect of a ban', 'difference')]:
    rows.append([label, welfare[0][key], welfare[1][key]])
table(['Point calculation', 'Concentrated values', 'Upper-tail mixture'],
      [[row[0], f'{row[1]:.12f}', f'{row[2]:.12f}'] for row in rows])
records(['ban_welfare_examples_floating'])

Point calculation,Concentrated values,Upper-tail mixture
Probing price,0.510980593208,0.515613314494
Knockout price,0.554378791664,0.640626332049
Rejected attempt,0.056313655095,0.066739617301
Early agreement,0.154353208514,0.068388497142
Avoided offer costs,0.004213337272,0.002702562289
Allocative gain,0.003883522055,0.002757306596
Forgone resolution gains,0.011624510881,0.005156517170
Welfare effect of a ban,-0.003527651553,0.000303351715


### Certified signs

The separate Arb calculation uses 60 decimal digits and 24,000 integration cells. It encloses a price fixed point in each box and checks equilibrium and refinement margins. The intervals below are rounded outwards to nine decimal places from the endpoint records. They exclude zero in opposite directions. The many digits in the point calculation above should not be mistaken for equally tight certified bounds.

In [4]:
sys.path.insert(0, str(ROOT / 'code'))
from decimal import localcontext
from generate_aligned_certificate_appendix import bounds
certificate = json.loads((RUN / 'ban_welfare_examples_interval.json').read_text())
with localcontext() as context:
    context.prec = 100
    intervals = [(name, bounds(case['welfare']['W_B_minus_W_D'], 9))
                 for name, case in certificate['cases'].items()]
table(['Calibration', 'Certified lower bound', 'Certified upper bound'],
      [[name, f'{lo:f}', f'{hi:f}'] for name, (lo, hi) in intervals])
records(['ban_welfare_examples_interval'])

Calibration,Certified lower bound,Certified upper bound
symmetric,-0.003662070,-0.003393114
upper_tail,0.000261675,0.000345072


## 4. Additional equilibria and calibrations

**Online appendix OA1 and OA5.** These exercises vary seller information and describe further equilibrium regimes. The aligned-composite example changes participation, fallback value and seller resolution benefit together; it is distinct from the main paper's participation-only illustration. Its probing-only calculation and three-action calculation also use different benefit distributions. The waiting–knockout result has no rejections and completes the regime analysis.

The full-menu certificate supports Tables OA5.2–OA5.3. The exact interval records are also converted back into the distributed LaTeX tables below; equality is checked byte for byte. The analytical arguments, including their equilibrium-selection restrictions, remain in the papers.

In [5]:
records(['aligned_composite_full_menu_interval', 'aligned_composite_calibration_floating',
         'urgency_only_knockout_gate_interval', 'urgency_only_knockout_gate_floating',
         'seller_urgency_state_floating'])
from generate_aligned_certificate_appendix import render
aligned = json.loads((RUN / 'aligned_composite_full_menu_interval.json').read_text())
with localcontext() as context:
    context.prec = 100
    generated_table = render(aligned)
if generated_table.encode('utf-8') != (ROOT / 'figures/aligned_certificate_tables.tex').read_bytes():
    raise RuntimeError('The distributed certificate tables differ from the fresh computation.')
print('Tables OA5.2–OA5.3: regenerated exactly from the fresh certificate.')

Tables OA5.2–OA5.3: regenerated exactly from the fresh certificate.


## 5. Disclosure, insurance and alternative bargaining protocols

The following checks accompany separate extensions; their assumptions are not imposed on the main model.

**OA2: disclosure.** The calculations examine verifiable participation information, including the opposite welfare rankings in Table OA2.1. They do not make deep preference parameters verifiable.

In [6]:
records(['demand_disclosure_floating', 'disclosure_welfare_examples_floating'])

**OA3: risk aversion.** The interval certificate verifies the binary heterogeneous-CARA calibration supporting Proposition OA3.1. It complements the analytical argument for nearby continuous types; it is not a numerical certificate for every continuous distribution.

In [7]:
records(['binary_cara_interval'])

**OA4: alternative protocols.** Exact arithmetic checks the continued pre-auction bargaining example and its tail, cost and welfare conditions. A separate interval calculation checks post-bidding seller counteroffers with action linkage. These are distinct protocol extensions, with their own assumptions and equilibrium claims.

In [8]:
records(['preemption_bargaining_equilibrium_exact', 'preemption_bargaining_tail_exact',
         'preemption_bargaining_cost_region_exact', 'patient_initiator_welfare_exact',
         'full_linkage_counteroffer_interval'])

## 6. Figures and numerical inputs

The figure checks regenerate CSV and TikZ outputs in isolated folders and compare them byte for byte. They include the full-menu price geometry, the composite probing strip, and the distributions of values and resolution benefits. Some generated diagrams are retained from the longer paper. Schematic timing and payoff diagrams drawn directly in LaTeX are not additional numerical exercises.

In [9]:
records(['maintained_theory_figures_roundtrip', 'numerical_distribution_figures_roundtrip'])
table(['Reproduced figure data/source'], [[path] for task in specs.values()
                                        if task['kind'] == 'figure_roundtrip'
                                        for path in task['outputs']])

Reproduced figure data/source
figures/probing_only_regime_strip.csv
figures/probing_only_outcomes.csv
figures/renegotiation_thresholds.csv
figures/theory_figure_metadata.json
figures/full_menu_outcomes_figure.tex
figures/full_menu_price_geometry_figure.tex
figures/probing_only_regime_figure.tex
figures/ban_value_distributions.csv
figures/ban_value_distributions_figure.tex
figures/numerical_urgency_distributions.csv


## 7. Research diagnostics

These four checks are retained for transparency and further work. **They are not extra certified results in the revised paper.** They examine the boundary of a simplified counteroffer rule, candidate continuous buyer-risk and local seller-risk calibrations, and an active post-bidding bargaining candidate. A diagnostic can pass by documenting a limitation; its status does not certify an equilibrium. Private offers during an ongoing auction belong to a separate archived research direction and are not part of the revised papers or this release.

In [10]:
records(['counteroffer_firewall_floating', 'continuous_cara_floating',
         'seller_cara_local_floating', 'active_postbidding_bargaining_diagnostic'])
if shown != set(tasks):
    raise RuntimeError('Workbook coverage differs from the executed task inventory.')
print(f'Coverage complete: all {len(tasks)} executed checks have an explanatory home above.')
print('Fresh results: ' + RUN.relative_to(ROOT).as_posix())

Coverage complete: all 22 executed checks have an explanatory home above.
Fresh results: code/reproduction_outputs/workbook-20260920T093823.408681Z


## 8. Inspect or extend the evidence

The downloadable ZIP is deliberately a minimal reproduction package. It contains the executable notebook, runnable source closure, configuration, pinned dependencies and expected figure sources; `PACKAGE_MANIFEST.json` records the SHA-256 hash of every other member. It excludes the PDFs, recorded execution logs and historical proof notes. The public workbook preserves the recorded results and links to the companion papers. Existing claim identifiers and some script docstrings retain the long manuscript's labels; the section references in this workbook refer to the revised pair.

To vary a calibration, edit the authoritative script named in its expanded record and run the notebook again. A changed calibration may fail the maintained equilibrium or refinement conditions. New outputs should be interpreted using those conditions rather than compared only by their welfare sign. This workbook reproduces the paper's internal numerical exercises; empirical figures quoted from other studies remain evidence from those cited sources.

For a compact account of the main formulas and model inputs, see `code/PAPER_CALCULATIONS.md` in the ZIP. The notebook source is generated from `code/NUMERICAL_WORKBOOK.md`; all economic calculations remain in the existing Python scripts.